# Benchmark del Entregable 2: entrenamiento en Colab o Kaggle

Este cuaderno prepara el entorno, ejecuta el benchmark y deja las salidas en una ubicación persistente.

- **Colab:** sube `trashnet-e2.zip` a la raíz de Google Drive. Las corridas terminadas se copian a `MyDrive/trashnet-e2-salidas/` en cuanto acaban, de modo que una desconexión solo pierde la corrida en curso.
- **Kaggle:** sube `trashnet-e2.zip` como Dataset, añádelo al cuaderno y activa *Internet* y la GPU. Las salidas quedan en `/kaggle/working` al guardar la versión (*Save Version → Save & Run All*).

Para repartir el trabajo entre máquinas, cada una ejecuta un subconjunto distinto de modelos (`MODELOS`). Al final se juntan las carpetas `results/` y se ejecuta la etapa `summary`.

## 1. Parámetros de esta sesión

In [ ]:
MODELOS = ["ViT-B16", "GFN"]
ETAPA = "all"
ACCUM = 1
ZIP_EN_DRIVE = "trashnet-e2.zip"
CARPETA_SALIDAS_DRIVE = "trashnet-e2-salidas"

`MODELOS` admite cualquier subconjunto de: `Celik-HC`, `MbXception`, `ViT-B16`, `GFN`, `FocusRCNet`, `ResMsCapsule`, `RWCNet`.

`ETAPA`: `estimate` (mide segundos por época y estima horas), `all` (búsqueda de hiperparámetros, semillas finales, resumen y empaquetado), `ablation`, `replicate`, `gradcam` (recalcula los mapas de Grad-CAM que falten o estén vacíos) o `summary`.

`ACCUM`: acumulación de gradiente. Usa 2 si aparece un error de memoria en la GPU; el lote efectivo sigue siendo 32.

## 2. Entorno

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
EN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content")
    ZIP = Path("/content/drive/MyDrive") / ZIP_EN_DRIVE
    RESPALDO = Path("/content/drive/MyDrive") / CARPETA_SALIDAS_DRIVE / "results"
elif EN_KAGGLE:
    BASE = Path("/kaggle/working")
    ZIP = next(Path("/kaggle/input").rglob("trashnet-e2.zip"), None)
    RESPALDO = None
else:
    BASE = Path.cwd().parent
    ZIP = None
    RESPALDO = None

PROYECTO = BASE / "trashnet-e2"
if not (PROYECTO / "src").exists():
    if ZIP is not None and ZIP.exists():
        with zipfile.ZipFile(ZIP) as z:
            z.extractall(BASE)
    elif EN_KAGGLE and next(Path("/kaggle/input").rglob("src/runner.py"), None):
        shutil.copytree(next(Path("/kaggle/input").rglob("src/runner.py")).parents[1], PROYECTO)
    else:
        raise FileNotFoundError("No se encontró trashnet-e2.zip. Revisa ZIP_EN_DRIVE o el Dataset de Kaggle.")

RESULTADOS = BASE / "results"
os.environ["TRASHNET_RESULTS"] = str(RESULTADOS)
os.environ["TRASHNET_DATA"] = str(BASE / "trashnet_data")
os.environ["TRASHNET_ACCUM"] = str(ACCUM)
if RESPALDO is not None:
    RESPALDO.mkdir(parents=True, exist_ok=True)
    os.environ["TRASHNET_BACKUP"] = str(RESPALDO)
    if any(RESPALDO.iterdir()):
        shutil.copytree(RESPALDO, RESULTADOS, dirs_exist_ok=True)
        print(f"Corridas previas restauradas desde {RESPALDO}")

os.chdir(PROYECTO)
print("Proyecto:", PROYECTO)
print("Resultados:", RESULTADOS)
print("Respaldo:", RESPALDO or "no (Kaggle guarda /kaggle/working al guardar la versión)")

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import torch
print(torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

## 3. Ejecución

La primera ejecución descarga TrashNet (43 MB) y los pesos de ImageNet, verifica el hash de cada imagen y la huella del conjunto de prueba, y escribe el preregistro. Si la sesión se corta, vuelve a ejecutar las celdas: las corridas terminadas se saltan.

In [ ]:
cmd = [sys.executable, "-m", "src.runner", ETAPA]
if ETAPA not in ("summary", "replicate"):
    cmd += ["--models", *MODELOS]
print(" ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
print("Código de salida:", proc.wait())

In [ ]:
subprocess.run([sys.executable, "-m", "src.runner", "status", "--models", *MODELOS])

## 4. Empaquetado de las salidas

Genera `resultados_entrega2_<fecha>.zip` con todas las salidas excepto los checkpoints de reanudación. En Colab queda en `MyDrive/trashnet-e2-salidas/`; en Kaggle, en `/kaggle/working/`.

In [ ]:
subprocess.run([sys.executable, "-m", "src.runner", "package"], check=True)
destino = RESPALDO if RESPALDO is not None else BASE
for f in sorted(Path(destino).rglob("resultados_entrega2_*.zip")) + sorted(BASE.glob("resultados_entrega2_*.zip")):
    print(f, f"{f.stat().st_size / 2**20:.1f} MB")